# 딥러닝실습 팀 프로젝트 — Video Modality

팀: 5조 | 담당: 24011917 이지현  
목표: CMU-MOSI 감성 분석 | 모달리티: 영상(이미지)  
아키텍처: **MTCNN** → EfficientNet-B0 / ResNet-18 → **768-dim feature** (멀티모달 fusion용)

# Part 1. 셋업

In [ ]:
import os, random, pickle, time, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet18, ResNet18_Weights,
)
from facenet_pytorch import MTCNN

import torch.multiprocessing as mp
mp.set_start_method('spawn', force=True)

print('torch:', torch.__version__)
print('CUDA :', torch.cuda.is_available())

In [ ]:
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 64
FEATURE_DIM = 768  # 멀티모달 fusion용 출력 차원

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ROOT_PATH    = r'C:\Users\user\Documents\50.2026\53.DL_실습\multimodal\data'
VIDEO_FOLDER = os.path.join(ROOT_PATH, 'Video', 'Segmented')
LABEL_PATH   = os.path.join(ROOT_PATH, 'mosi_text_metadata.csv')
PICKLE_PATH  = os.path.join(ROOT_PATH, 'video_preprocessed.pkl')

print(f'device: {device}')
print(f'VIDEO_FOLDER: {VIDEO_FOLDER}')
print(f'PICKLE_PATH: {PICKLE_PATH}')

# Part 2. 라벨 로드

In [ ]:
label_df = pd.read_csv(LABEL_PATH)
label_df = label_df.dropna(subset=['label'])  # 텍스트와 동일하게 dropna

# ✅ 텍스트와 동일: CSV 행 순서 그대로 리스트로 보관
video_keys_ordered = [
    f"{r['video_id']}_{r['seg_idx']}"
    for _, r in label_df.iterrows()
]
labels_ordered = [int(r['label']) for _, r in label_df.iterrows()]

# 기존 코드 호환용 dict도 유지
label_dict = dict(zip(video_keys_ordered, labels_ordered))

print(f'총 라벨 수: {len(video_keys_ordered)}')
print(f"긍정: {sum(v==1 for v in labels_ordered)}  "
      f"부정: {sum(v==0 for v in labels_ordered)}")

# Part 2. 전처리 (MTCNN + Pickle 캐싱)

- **MTCNN**: facenet_pytorch 기반 딥러닝 얼굴 탐지 (Haar Cascade 대비 정확도 향상)
- fallback: MTCNN 탐지 실패 시 전체 이미지를 224×224로 리사이즈
- 전처리 결과를 Pickle로 저장해 재실행 시 즉시 로드

In [ ]:
# 캐시 로드 or 새로 생성
if os.path.exists(PICKLE_PATH):
    print('캐시 로드:', PICKLE_PATH)
    with open(PICKLE_PATH, 'rb') as f:
        images, labels = pickle.load(f)
    print(f'이미지: {images.shape}  라벨: {labels.shape}')
    print(f"긍정: {(labels==1).sum()}  부정: {(labels==0).sum()}")
else:
    print('피클 없음 → build_dataset() 실행 필요')
    print('아래 셀을 실행하세요')

In [ ]:
mtcnn = MTCNN(image_size=IMG_SIZE, margin=20, keep_all=False,
              device=device, post_process=False)
MTCNN_INPUT_SIZE = (640, 480)

def extract_middle_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(total // 2, 0))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

def build_dataset():
    """
    ✅ 텍스트와 동일한 방식:
       CSV 행 순서(video_keys_ordered) 그대로 순회.
       os.listdir() 완전 제거 → 순서 보장.
    """
    face_found = face_missed = 0
    frames_for_mtcnn, valid_keys_list, valid_labels_list = [], [], []

    for key, lbl in tqdm(zip(video_keys_ordered, labels_ordered),
                         total=len(video_keys_ordered), desc='프레임 추출'):
        video_path = os.path.join(VIDEO_FOLDER, key + '.mp4')
        if not os.path.exists(video_path):
            continue
        img_pil = extract_middle_frame(video_path)
        if img_pil is None:
            continue
        frames_for_mtcnn.append(img_pil.resize(MTCNN_INPUT_SIZE, Image.BILINEAR))
        valid_keys_list.append(key)
        valid_labels_list.append(lbl)

    MTCNN_BATCH = 64
    images, labels_out = [], []
    for i in tqdm(range(0, len(frames_for_mtcnn), MTCNN_BATCH), desc='MTCNN 배치'):
        batch_frames = [f.resize(MTCNN_INPUT_SIZE, Image.BILINEAR)
                        for f in frames_for_mtcnn[i:i+MTCNN_BATCH]]
        batch_lbls   = valid_labels_list[i:i+MTCNN_BATCH]
        try:
            faces = mtcnn(batch_frames)
        except Exception as e:
            print(f'[WARN] 배치 MTCNN 실패: {e}')
            faces = [None] * len(batch_frames)
        for face, orig_pil, lbl in zip(faces, batch_frames, batch_lbls):
            if face is not None:
                img_np = face.clamp(0, 255).permute(1, 2, 0).byte().cpu().numpy()
                face_found += 1
            else:
                img_np = np.array(orig_pil.resize((IMG_SIZE, IMG_SIZE)))
                face_missed += 1
            images.append(img_np)
            labels_out.append(lbl)

    print(f'얼굴 발견: {face_found}  얼굴 누락: {face_missed}')
    return np.array(images), np.array(labels_out)

# 캐시 로드 or 새로 생성
if os.path.exists(PICKLE_PATH):
    print('캐시 로드:', PICKLE_PATH)
    with open(PICKLE_PATH, 'rb') as f:
        images, labels = pickle.load(f)
else:
    images, labels = build_dataset()
    with open(PICKLE_PATH, 'wb') as f:
        pickle.dump((images, labels), f)
    print('피클 저장 완료:', PICKLE_PATH)

print(f'이미지: {images.shape}  라벨: {labels.shape}')
print(f'긍정: {(labels==1).sum()}  부정: {(labels==0).sum()}')

# Part 3. DataLoader

In [ ]:
class GaussianNoise:
    """Tensor에 가우시안 노이즈를 추가하는 커스텀 transform."""
    def __init__(self, std=0.05):
        self.std = std
    def __call__(self, t):
        return (t + torch.randn_like(t) * self.std).clamp(0, 1)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    GaussianNoise(std=0.05),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

#  train 64% / val 16% / test 20%
X_tr, X_test, y_tr, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=SEED, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr
)

g = torch.Generator(); g.manual_seed(SEED)

NW = 0
train_loader = DataLoader(ImageDataset(X_train, y_train, train_tf),
                          batch_size=BATCH_SIZE, shuffle=False,
                          generator=g, num_workers=NW,
                          pin_memory=True)
val_loader   = DataLoader(ImageDataset(X_val, y_val, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=True)
test_loader  = DataLoader(ImageDataset(X_test, y_test, test_tf),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=True)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'긍정/부정 비율 (Train): {(y_train==1).sum()}/{(y_train==0).sum()}')
print(f'긍정/부정 비율 (Val): {(y_val==1).sum()}/{(y_val==0).sum()}')
print(f'긍정/부정 비율 (Test): {(y_test==1).sum()}/{(y_test==0).sum()}')
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}')

# Part 4. 모델 정의 — VideoEncoder

efficientnet: 1280-dim → 768-dim  
resnet18: 512-dim → 768-dim

- `feature_only=False` (기본): 분류 logit 반환 → 단독 학습/평가
- `feature_only=True`: 768-dim 벡터 반환 → 멀티모달 fusion 입력

In [ ]:
class VideoEncoder(nn.Module):
    """EfficientNet-B0 또는 ResNet-18 기반 영상 인코더.
    feature_only=True → 768-dim 벡터 (멀티모달 fusion용)."""

    def __init__(self, backbone='efficientnet', num_classes=2, feature_dim=FEATURE_DIM):
        super().__init__()
        if backbone == 'efficientnet':
            base   = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            in_dim = base.classifier[1].in_features   # 1280
            base.classifier = nn.Identity()
        elif backbone == 'resnet18':
            base   = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
            in_dim = base.fc.in_features              # 512
            base.fc = nn.Identity()
        else:
            raise ValueError(f'Unknown backbone: {backbone}')

        self.backbone   = base
        self.proj       = nn.Sequential(
            nn.Linear(in_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x, feature_only=False):
        feat = self.proj(self.backbone(x))   # (B, 768)
        return feat if feature_only else self.classifier(feat)

    def extract_features(self, loader, device):
        """전체 데이터셋 768-dim 피처 추출 (멀티모달용)."""
        self.eval()
        feats, ys = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc='feature extraction'):
                feats.append(self.forward(x.to(device), feature_only=True).cpu())
                ys.append(y)
        return torch.cat(feats), torch.cat(ys)


for name, bk in [('EfficientNet-B0', 'efficientnet'), ('ResNet-18', 'resnet18')]:
    m     = VideoEncoder(backbone=bk)
    total = sum(p.numel() for p in m.parameters())
    print(f'{name}: {total:,} params')
    del m

# Part 5. 랜덤 서치

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out  = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / total, correct / total


def run_search(backbone, param_space, n_search, train_loader, val_loader, device):
    """val_loader로 하이퍼파라미터 탐색 (test leakage 방지).
    최고 성능 모델 state_dict 저장 후 반환."""
    criterion = nn.CrossEntropyLoss()
    all_combos = [
        (lr, ep, opt, wd)
        for lr  in param_space['lr']
        for ep  in param_space['num_epochs']
        for opt in param_space['optimizer']
        for wd  in param_space['weight_decay']
    ]
    configs  = random.sample(all_combos, min(n_search, len(all_combos)))
    results  = []
    best_acc, best_cfg, best_state = 0.0, {}, None

    for i, (lr, ep, opt_name, wd) in enumerate(configs):
        model = VideoEncoder(backbone=backbone).to(device)
        optimizer = (
            optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
            if opt_name == 'Adam'
            else optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
        )
        t0 = time.time()
        for ep_i in range(1, ep + 1):
            _, tr_acc  = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
            _, val_acc = run_epoch(model, val_loader,   criterion, None,      device, train=False)
            print(f'  Epoch {ep_i}/{ep}  train={tr_acc:.4f}  val={val_acc:.4f}')
        elapsed = time.time() - t0

        print(f'[{backbone}] {i+1}/{len(configs)} '
              f'lr={lr} ep={ep} {opt_name} wd={wd} '
              f'→ val_acc={val_acc:.4f} ({elapsed:.0f}s)')

        rec = {'backbone': backbone, 'lr': lr, 'num_epochs': ep,
               'optimizer': opt_name, 'weight_decay': wd,
               'val_acc': round(val_acc, 4)}
        results.append(rec)
        if val_acc > best_acc:
            best_acc, best_cfg = val_acc, rec.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        del model; torch.cuda.empty_cache()

    return pd.DataFrame(results).sort_values('val_acc', ascending=False), best_cfg, best_state

In [ ]:
PARAM_SPACE = {
    'lr'          : [5e-4, 3e-4, 1e-4],
    'num_epochs'  : [10, 15, 20],
    'optimizer'   : ['Adam'],
    'weight_decay': [1e-3, 5e-3],
}
N_SEARCH = 10

random.seed(SEED)
print('=== EfficientNet-B0 Random Search ===')
eff_results, eff_best, eff_best_state = run_search(
    'efficientnet', PARAM_SPACE, N_SEARCH, train_loader, val_loader, device
)
print(f'\n★ EfficientNet-B0 최적 (val): {eff_best}')

In [ ]:
random.seed(SEED)
print('=== ResNet-18 Random Search ===')
res_results, res_best, res_best_state = run_search(
    'resnet18', PARAM_SPACE, N_SEARCH, train_loader, val_loader, device
)
print(f'\n★ ResNet-18 최적 (val): {res_best}')

# Part 5.5. 기본 모델 + 조기종료(Early Stopping) 비교

랜덤서치 없이 기본 하이퍼파라미터 + EarlyStopping 적용
- `patience=5`: val_acc가 5 epoch 동안 개선 없으면 학습 중단  
- optimizer: Adam lr=3e-4, weight_decay=1e-3  
- 최대 epoch: 50

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience      = patience
        self.min_delta     = min_delta
        self.best_val      = -float('inf')
        self.counter       = 0
        self.best_state    = None
        self.stopped_epoch = 0

    def step(self, val_acc, model):
        if val_acc > self.best_val + self.min_delta:
            self.best_val   = val_acc
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience


def train_with_earlystop(backbone, lr=3e-4, weight_decay=1e-3,
                         max_epochs=50, patience=5,
                         train_loader=train_loader,
                         val_loader=val_loader,
                         test_loader=test_loader,
                         device=device):
    model     = VideoEncoder(backbone=backbone).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    stopper   = EarlyStopping(patience=patience)

    train_accs, val_accs, test_accs = [], [], []

    for ep in range(1, max_epochs + 1):
        _, tr_acc  = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        _, val_acc = run_epoch(model, val_loader,   criterion, None,      device, train=False)
        _, te_acc  = run_epoch(model, test_loader,  criterion, None,      device, train=False)
        scheduler.step()

        train_accs.append(tr_acc)
        val_accs.append(val_acc)
        test_accs.append(te_acc)

        should_stop = stopper.step(val_acc, model)

        print(f'  Epoch {ep:02d}/{max_epochs}  '
              f'train={tr_acc:.4f}  val={val_acc:.4f}  test={te_acc:.4f}'
              + ('  ← best' if stopper.counter == 0 else
                 f'  (patience {stopper.counter}/{patience})'))

        if should_stop:
            stopper.stopped_epoch = ep
            print(f'  ★ EarlyStopping @ epoch {ep}  best_val={stopper.best_val:.4f}')
            break

    if stopper.stopped_epoch == 0:
        stopper.stopped_epoch = max_epochs

    model.load_state_dict({k: v.to(device) for k, v in stopper.best_state.items()})
    print(f'  → 최종 best_val={stopper.best_val:.4f}  실제 종료 epoch={stopper.stopped_epoch}')
    return model, train_accs, val_accs, test_accs, stopper.stopped_epoch

In [ ]:
print('=== [Baseline + EarlyStopping] EfficientNet-B0 ===')
es_eff_model, es_eff_tr, es_eff_val, es_eff_te, es_eff_stop = train_with_earlystop(
    backbone='efficientnet', lr=3e-4, weight_decay=1e-3, max_epochs=50, patience=8
)

In [ ]:
print('=== [Baseline + EarlyStopping] ResNet-18 ===')
es_res_model, es_res_tr, es_res_val, es_res_te, es_res_stop = train_with_earlystop(
    backbone='resnet18', lr=3e-4, weight_decay=1e-3, max_epochs=50, patience=5
)

In [ ]:
# EarlyStopping 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, name, tr, val, te, stop_ep in [
    (axes[0], 'EfficientNet-B0 (ES)', es_eff_tr, es_eff_val, es_eff_te, es_eff_stop),
    (axes[1], 'ResNet-18 (ES)',        es_res_tr, es_res_val, es_res_te, es_res_stop),
]:
    eps = range(1, len(tr) + 1)
    ax.plot(eps, tr,  label='Train')
    ax.plot(eps, val, label='Val',  linestyle='--')
    ax.plot(eps, te,  label='Test', linestyle=':')
    ax.axvline(x=stop_ep, color='red', linestyle='-.', linewidth=1.2, label=f'ES stop (ep {stop_ep})')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(True)

plt.suptitle('Baseline + EarlyStopping Learning Curves', fontsize=13)
plt.tight_layout()
plt.savefig('viz_earlystop_curves.png', dpi=150)
plt.show()

In [ ]:
all_results = pd.concat([eff_results, res_results]).reset_index(drop=True)
print(all_results.to_string(index=False))

# Part 6. 최적 파라미터로 최종 학습

In [ ]:
def train_full(backbone, cfg, train_loader, val_loader, test_loader, device, extra_epochs=10):
    """best_state 로드 없이 처음부터 재학습 — epoch은 cfg + extra_epochs."""
    model = VideoEncoder(backbone=backbone).to(device)
    criterion = nn.CrossEntropyLoss()
    wd = cfg.get('weight_decay', 0)
    optimizer = (
        optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=wd)
        if cfg['optimizer'] == 'Adam'
        else optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=wd)
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['num_epochs'] + extra_epochs
    )

    total_epochs = cfg['num_epochs'] + extra_epochs
    train_accs, val_accs, test_accs = [], [], []
    best_val, best_state = 0.0, None

    for ep in range(1, total_epochs + 1):
        _, tr_acc  = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        _, val_acc = run_epoch(model, val_loader,   criterion, None,      device, train=False)
        _, te_acc  = run_epoch(model, test_loader,  criterion, None,      device, train=False)
        scheduler.step()
        train_accs.append(tr_acc); val_accs.append(val_acc); test_accs.append(te_acc)
        print(f'  Epoch {ep}/{total_epochs}  train={tr_acc:.4f}  val={val_acc:.4f}  test={te_acc:.4f}')

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model, train_accs, val_accs, test_accs

In [ ]:
print(f'=== EfficientNet-B0 최종 학습 ===')
print(f'config: {eff_best}')
eff_model, eff_tr_accs, eff_val_accs, eff_te_accs = train_full(
    'efficientnet', eff_best, train_loader, val_loader, test_loader, device
)

In [ ]:
print(f'=== ResNet-18 최종 학습 ===')
print(f'config: {res_best}')
res_model, res_tr_accs, res_val_accs, res_te_accs = train_full(
    'resnet18', res_best, train_loader, val_loader, test_loader, device
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, tr, val, te in [
    (axes[0], 'EfficientNet-B0', eff_tr_accs, eff_val_accs, eff_te_accs),
    (axes[1], 'ResNet-18',       res_tr_accs, res_val_accs, res_te_accs),
]:
    eps = range(1, len(tr) + 1)
    ax.plot(eps, tr,  label='Train')
    ax.plot(eps, val, label='Val',  linestyle='--')
    ax.plot(eps, te,  label='Test', linestyle=':')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('viz_accuracy_curves.png', dpi=150)
plt.show()

In [ ]:
label_names = ['Negative', 'Positive']

for name, model in [('EfficientNet-B0', eff_model), ('ResNet-18', res_model)]:
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())
    print(f'=== {name} (test_acc={sum(p==t for p,t in zip(preds,targets))/len(targets):.4f}) ===')
    print(classification_report(targets, preds, target_names=label_names))

# Part 7. 768-dim & 256-dim 피처 추출 (멀티모달 Fusion 준비)

`VideoEncoder.extract_features()` 로 768-dim 벡터를 추출한다.  
멀티모달 학습 시 이 벡터를 audio/text 피처와 concat하여 MLP에 입력한다.

```
video 768-dim ─┐
audio 768-dim ─┼─ concat(2304) → MLP → 감성 분류
text  768-dim ─┘
```

In [ ]:
# val_acc 기준으로 모델 선택
if max(eff_val_accs) >= max(res_val_accs):
    best_model_name, best_model = 'EfficientNet-B0', eff_model
else:
    best_model_name, best_model = 'ResNet-18', res_model

final_val_acc  = max(eff_val_accs  if best_model_name == 'EfficientNet-B0' else res_val_accs)
final_test_acc = max(eff_te_accs   if best_model_name == 'EfficientNet-B0' else res_te_accs)
print(f'선택된 모델: {best_model_name} (val_acc={final_val_acc:.4f}  test_acc={final_test_acc:.4f})')

# 전체 데이터셋 768-dim 피처 추출
all_loader = DataLoader(
    ImageDataset(images, labels, test_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NW, persistent_workers=False
)
video_feats_768, video_labels = best_model.extract_features(all_loader, device)
print(f'768-dim 피처 shape: {video_feats_768.shape}')

# 피처 분포 시각화
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(video_feats_768.numpy().flatten(), bins=60, color='steelblue', alpha=0.7)
ax.set_title(f'{best_model_name} 768-dim feature distribution')
ax.set_xlabel('Feature value'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('viz_feature_dist.png', dpi=150)
plt.show()

# 768-dim 피처 저장
feat_save_path_768 = os.path.join(ROOT_PATH, 'video_features_768.pkl')
with open(feat_save_path_768, 'wb') as f:
    pickle.dump({'features': video_feats_768, 'labels': video_labels}, f)
print(f'768-dim 피처 저장: {feat_save_path_768}')

In [ ]:
# 256-dim 피처 추출 (proj_256 레이어 추가)
class VideoEncoder256(nn.Module):
    """기존 VideoEncoder에 256-dim 프로젝션 추가."""
    def __init__(self, base_encoder):
        super().__init__()
        self.encoder = base_encoder
        self.proj_256 = nn.Sequential(
            nn.Linear(FEATURE_DIM, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

    def extract_features_256(self, loader, device):
        self.eval()
        feats, ys = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc='extract 256-dim'):
                feat_768 = self.encoder(x.to(device), feature_only=True)
                feat_256 = self.proj_256(feat_768)
                feats.append(feat_256.cpu())
                ys.append(y)
        return torch.cat(feats), torch.cat(ys)

# 기존 best_model 위에 256-dim 프로젝션 헤드만 추가 (백본은 학습된 상태 유지)
enc256 = VideoEncoder256(best_model).to(device)
video_feats_256, _ = enc256.extract_features_256(all_loader, device)
print(f'256-dim 피처 shape: {video_feats_256.shape}')

feat_save_path_256 = os.path.join(ROOT_PATH, 'video_features_256.pkl')
with open(feat_save_path_256, 'wb') as f:
    pickle.dump({'features': video_feats_256, 'labels': video_labels}, f)
print(f'256-dim 피처 저장: {feat_save_path_256}')